In [29]:
import pandas as pd
import pycountry

In [30]:
def normalize_country(name):
    if pd.isna(name):
        return "XX"

    # Clean up whitespace and force upper case for comparison
    name_clean = str(name).strip()

    try:
        # Look up by name, alpha_3 (USA), or alpha_2 (US)
        country = pycountry.countries.lookup(name_clean)
        return country.alpha_2  # Returns the 2-letter uppercase code
    except LookupError:
        # If pycountry can't find a match, default to XX
        return "XX"

In [73]:
def transform_data(df):

    # Identify malformed / quarantined rows
    # Critical fields that MUST be parseable to be considered a valid row
    parsed_event_time = pd.to_datetime(df['event_time'], errors='coerce', utc=True)
    parsed_ingested_at = pd.to_datetime(df['ingested_at'], errors='coerce', utc=True)
    
    # Definition of a bad/malformed row:
    # - Missing event_id
    # - Completely unparseable timestamps (NaT)
    is_malformed = (
        df['event_id'].isna() | 
        parsed_event_time.isna() | 
        parsed_ingested_at.isna()
    )

    # Separate good rows from quarantined rows
    df_quarantine = df[is_malformed].copy()
    df_clean = df[~is_malformed].copy()

    # revenue_usd as a float; unparseable and empty values become 0.0, but count them and report the count
    cleaned_strings = (
        df_clean["revenue_usd"].astype(str).str.replace(",", ".").str.strip()
    )

    convert_dt = pd.to_numeric(cleaned_strings, errors="coerce") #forcing unparseable text/empty values to NaN

    cnt_unparseable_or_empty = convert_dt.isna().sum()

    df_clean['revenue_usd'] = convert_dt.fillna(0.0)

    print(f"Found {cnt_unparseable_or_empty} unparseable or empty values")


    # country normalised to an upper-case two-letter code; anything you cannot map becomes 'XX'
    df_clean["country"] = df_clean["country"].apply(normalize_country)


    # timestamps parsed as timezone-aware UTC
    df_clean['event_time'] = pd.to_datetime(df_clean['event_time'], utc=True)
    df_clean['ingested_at'] = pd.to_datetime(df_clean['ingested_at'], utc=True)


    # test rows dropped
    df_clean = df_clean[df_clean['is_test'] == False]


    # duplicates resolved with the same rule as in task 2.1.
    df_clean = df_clean.sort_values(by = ['event_id', 'ingested_at'], ascending = [True, True])

    df_clean = df_clean.drop_duplicates(subset=['event_id'], keep='last')

    print(f"Clean rows processed: {len(df_clean)} | Quarantined rows: {len(df_quarantine)}")
    return df_clean, df_quarantine

In [ ]:
file_path = '../data/events_raw.csv'

try:
    print(f"Loading data from {file_path}...")
    df_raw = pd.read_csv(file_path)
except Exception as e:
    print(f"Failed to read CSV: {e}")
    df_raw = pd.DataFrame()

if not df_raw.empty:
    df_clean_final, df_quarantine_final = transform_data(df_raw)

Loading data from ../data/events_raw.csv...
Found 182 unparseable or empty values
Clean rows processed: 187 | Quarantined rows: 0


In [77]:
df_clean_final

,event_id,user_id,app_id,event_name,event_time,ingested_at,country,media_source,campaign,revenue_usd,is_test
36,00623e64-4c8e-4367-8c73-d7af7413fc85,usr_f20f8639,app_101,registration,2026-09-01 06:14:50+00:00,2026-09-01 08:19:20+00:00,US,Facebook Ads,PP_AND_US_purchase_broad,0.00,False
164,00d7ded6-6545-4f8b-b85e-78012c130713,usr_41cd5213,app_401,install,2026-08-31 20:14:35+00:00,2026-09-03 03:54:44+00:00,US,Facebook Ads,BM_IOS_US_launch,0.00,False
137,018975dd-be2a-4d9d-9add-0733f2b9642a,usr_d60f097e,app_101,registration,2026-08-27 22:49:04+00:00,2026-08-27 22:49:39+00:00,US,organic,NaN,0.00,False
212,01b11584-7caa-4bea-9d4e-9d6ff503eb65,usr_81376ee9,app_201,install,2026-08-26 19:03:31+00:00,2026-09-01 08:58:23+00:00,US,organic,NaN,0.00,False
66,01c358e0-2e9f-4408-89bb-cb71a095bd5d,usr_d60f097e,app_101,install,2026-08-27 22:38:00+00:00,2026-08-27 22:40:43+00:00,US,organic,NaN,0.00,False
11,02643b20-2ce4-4afe-9bb4-46a49728585e,usr_b1cafebd,app_102,purchase,2026-09-02 04:20:46+00:00,2026-09-07 11:53:56+00:00,XX,organic,NaN,4.99,False
241,033bed1d-ef24-401a-98d5-b7ec298e33d9,usr_8b5e3449,app_102,purchase,2026-09-02 01:24:39+00:00,2026-09-06 10:49:52+00:00,MX,organic,NaN,4.99,False
201,0383200a-2589-443b-acdf-ef0edc267e93,usr_5c0c5047,app_201,purchase,2026-08-31 13:42:42+00:00,2026-08-31 16:54:50+00:00,US,organic,NaN,0.00,False
104,0530dad0-3452-4c81-9e0b-3799d9fe5a01,usr_796abe47,app_401,install,2026-08-31 09:01:28+00:00,2026-08-31 09:03:24+00:00,US,Facebook Ads,BM_IOS_US_launch,0.00,False
89,05426617-c18d-4bc3-9e90-932cf232ae34,usr_5c0c5047,app_201,subscription_start,2026-08-30 20:42:29+00:00,2026-09-01 23:06:59+00:00,US,organic,NaN,9.99,False


In [78]:
df_quarantine_final

,event_id,user_id,app_id,event_name,event_time,ingested_at,country,media_source,campaign,revenue_usd,is_test
